Import libraries

In [1]:
import pandas as pd
import numpy as np

Load datasets

In [3]:
df_filtered = pd.read_csv('/content/upi_fraud_dataset_Filtered.csv')
df_minmax   = pd.read_csv('/content/upi_fraud_dataset_minmax.csv')
df_std      = pd.read_csv('/content/upi_fraud_dataset_standardized.csv')
df_feat     = pd.read_csv('/content/upi_fraud_feature_engineered.csv')

Reset index

In [4]:
dfs = [df_filtered, df_minmax, df_std, df_feat]

for df in dfs:
    df.reset_index(drop=True, inplace=True)

df_filtered, df_minmax, df_std, df_feat = dfs

In [5]:
y = df_filtered['FraudFlag']

In [6]:
for df in [df_minmax, df_std, df_feat]:
    df.drop(['FraudFlag'], axis=1, inplace=True, errors='ignore')

In [7]:
df_minmax = df_minmax.add_prefix("minmax_")
df_std    = df_std.add_prefix("std_")
df_feat   = df_feat.add_prefix("feat_")

In [8]:
X = pd.concat([df_filtered, df_minmax, df_std, df_feat], axis=1)

In [9]:
X = X.loc[:, ~X.columns.duplicated()]

In [10]:
X = X.drop(['FraudFlag', 'TransactionID', 'UserID', 'PhoneNumber', 'IP_Address'], axis=1, errors='ignore')

In [11]:
if 'Timestamp' in X.columns:
    X['Timestamp'] = pd.to_datetime(X['Timestamp'], errors='coerce')
    X['Timestamp'] = X['Timestamp'].astype('int64') // 10**9

In [12]:
X = pd.get_dummies(X, drop_first=True)

In [13]:
X = X.fillna(0)

In [14]:
print("NaN count:", X.isnull().sum().sum())  # should be 0
print("Shape:", X.shape)

NaN count: 0
Shape: (11744, 81)


In [15]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [16]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

RandomForestClassifier(max_depth=10, n_estimators=200, n_jobs=-1,
                       random_state=42)

In [17]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nReport:\n", classification_report(y_test, y_pred))

Accuracy: 0.8467432950191571

Report:
               precision    recall  f1-score   support

           0       0.84      0.99      0.91      1811
           1       0.91      0.37      0.52       538

    accuracy                           0.85      2349
   macro avg       0.88      0.68      0.72      2349
weighted avg       0.86      0.85      0.82      2349



In [18]:
import joblib
joblib.dump(model, 'fraud_model.pkl')

['fraud_model.pkl']

In [20]:
from sklearn.metrics import classification_report, confusion_matrix

print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))


Classification Report:
               precision    recall  f1-score   support

           0       0.84      0.99      0.91      1811
           1       0.91      0.37      0.52       538

    accuracy                           0.85      2349
   macro avg       0.88      0.68      0.72      2349
weighted avg       0.86      0.85      0.82      2349


Confusion Matrix:
 [[1792   19]
 [ 341  197]]
